In [ ]:
!aws configure set aws_access_key_id #hided for security
!aws configure set aws_secret_access_key #hided for security
!aws configure set default.region us-east-1

In [2]:
import mlflow
# Step 2: Set up the MLflow tracking server
mlflow.set_tracking_uri("http://ec2-3-87-202-243.compute-1.amazonaws.com:5000")

In [9]:
# Set or create an experiment
mlflow.set_experiment("using_distilbert")

2026/03/03 00:01:01 INFO mlflow.tracking.fluent: Experiment with name 'using_distilbert' does not exist. Creating a new experiment.


<Experiment: artifact_location='s3://mlflow-tracking-bucket26/18', creation_time=1772476263002, experiment_id='18', last_update_time=1772476263002, lifecycle_stage='active', name='using_distilbert', tags={}>

In [3]:
import pandas as pd

df = pd.read_csv('youtube_preprocessing.csv').dropna()
df.shape

(3118, 3)

In [4]:
import mlflow
import mlflow.pytorch
import torch
import numpy as np
import pandas as pd

from datasets import Dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

c:\Users\Urmi Kanrar\anaconda3\envs\sentiment\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
label_mapping = {-1: 2, 0: 0, 1: 1}
df["label_fixed"] = df["sentiment_encoded"].map(label_mapping)

X = df["Comment"]
y = df["label_fixed"]
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [15]:
train_dataset = Dataset.from_dict({
    "text": X_train.tolist(),
    "label": y_train.tolist()
})

test_dataset = Dataset.from_dict({
    "text": X_test.tolist(),
    "label": y_test.tolist()
})

In [16]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

Map: 100%|██████████| 624/624 [00:00<00:00, 8405.82 examples/s]


In [17]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro"
    )

    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        labels, preds, average="weighted"
    )

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted
    }

In [18]:
num_labels = df["sentiment_encoded"].nunique()

with mlflow.start_run():

    model = DistilBertForSequenceClassification.from_pretrained(
        "distilbert-base-uncased",
        num_labels=num_labels
    )

    training_args = TrainingArguments(
        output_dir="./results",
        eval_strategy="epoch",
        save_strategy="no",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        weight_decay=0.01,
        logging_dir="./logs",
        logging_steps=50
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        compute_metrics=compute_metrics
    )

    trainer.train()

    eval_results = trainer.evaluate()

    from sklearn.metrics import classification_report

    # Get predictions
    predictions = trainer.predict(test_dataset)
    y_pred = np.argmax(predictions.predictions, axis=1)
    y_true = predictions.label_ids

    # Print classification report
    print("\nClassification Report:\n")
    print(classification_report(y_true, y_pred))

    # Log metrics
    for key, value in eval_results.items():
        if isinstance(value, float):
            mlflow.log_metric(key, value)

    # Log parameters
    mlflow.log_param("model_name", "distilbert-base-uncased")
    mlflow.log_param("epochs", 3)
    mlflow.log_param("batch_size", 16)
    mlflow.log_param("learning_rate", 2e-5)

    # Log model
    mlflow.pytorch.log_model(model, "distilbert_model")

    print("Final Results:", eval_results)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 543.98it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_D

Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro,Precision Weighted,Recall Weighted,F1 Weighted
1,0.814320,0.754779,0.689103,0.699666,0.621695,0.630768,0.703074,0.689103,0.679179
2,0.614461,0.682868,0.727564,0.721071,0.682690,0.695584,0.728662,0.727564,0.724731
3,0.479922,0.684491,0.719551,0.704797,0.688436,0.693990,0.721803,0.719551,0.718834


c:\Users\Urmi Kanrar\anaconda3\envs\sentiment\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\Users\Urmi Kanrar\anaconda3\envs\sentiment\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\Users\Urmi Kanrar\anaconda3\envs\sentiment\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


c:\Users\Urmi Kanrar\anaconda3\envs\sentiment\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



Classification Report:

              precision    recall  f1-score   support

           0       0.67      0.57      0.62        99
           1       0.77      0.74      0.76       316
           2       0.67      0.76      0.71       209

    accuracy                           0.72       624
   macro avg       0.70      0.69      0.69       624
weighted avg       0.72      0.72      0.72       624



2026/03/03 00:37:55 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\URMIKA~1\AppData\Local\Temp\tmp7caf4epk\model\data, flavor: pytorch). Fall back to return ['torch==2.10.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 
2026/03/03 00:37:55 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Final Results: {'eval_loss': 0.6844910383224487, 'eval_accuracy': 0.719551282051282, 'eval_precision_macro': 0.7047972592122878, 'eval_recall_macro': 0.6884361043255115, 'eval_f1_macro': 0.6939897702952816, 'eval_precision_weighted': 0.7218026119322092, 'eval_recall_weighted': 0.719551282051282, 'eval_f1_weighted': 0.7188336242125647, 'eval_runtime': 21.61, 'eval_samples_per_second': 28.876, 'eval_steps_per_second': 1.805, 'epoch': 3.0}


2026/03/03 00:43:43 INFO mlflow.tracking._tracking_service.client: 🏃 View run bold-cod-998 at: http://ec2-3-87-202-243.compute-1.amazonaws.com:5000/#/experiments/18/runs/b6c69037c25346ae8b8973ee8cf7ca7f.
2026/03/03 00:43:43 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://ec2-3-87-202-243.compute-1.amazonaws.com:5000/#/experiments/18.
